# Generate RetailHub training dataset

This notebook builds the foundation for the Medi course:

- Bronze/Silver/Gold schemas,
- synthetic RetailHub customers, products, orders and order lines,
- controlled data-quality issues for labs,
- starter Gold objects used by modules.

In [ ]:
%run ../setup/00_setup

In [ ]:
from pyspark.sql import functions as F

spark.sql(f"USE CATALOG {CATALOG}")

for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

print("Building dataset in", CATALOG)

## 1. Silver master data

In [ ]:
customers = (
    spark.range(1, 10001).withColumnRenamed("id", "customer_id")
    .withColumn("customer_name", F.concat(F.lit("Customer "), F.col("customer_id")))
    .withColumn("email", F.concat(F.lit("customer_"), F.col("customer_id"), F.lit("@retailhub.example")))
    .withColumn("city", F.expr("element_at(array('Warsaw','Krakow','Gdansk','Poznan','Wroclaw'), cast(customer_id % 5 as int) + 1)"))
    .withColumn("region", F.expr("element_at(array('North','South','East','West'), cast(customer_id % 4 as int) + 1)"))
    .withColumn("country", F.lit("PL"))
    .withColumn("segment", F.expr("element_at(array('Consumer','SMB','Enterprise'), cast(customer_id % 3 as int) + 1)"))
    .withColumn("created_date", F.date_add(F.to_date(F.lit("2022-01-01")), (F.col("customer_id") % 900).cast("int")))
)

products = (
    spark.range(1, 501).withColumnRenamed("id", "product_id")
    .withColumn("product_name", F.concat(F.lit("Product "), F.col("product_id")))
    .withColumn("category", F.expr("element_at(array('Bikes','Components','Clothing','Accessories','Services'), cast(product_id % 5 as int) + 1)"))
    .withColumn("subcategory", F.concat(F.col("category"), F.lit(" / "), F.expr("cast(product_id % 10 as string)")))
    .withColumn("unit_cost", F.round(F.lit(10) + (F.col("product_id") % 90) * 1.37, 2))
    .withColumn("unit_price", F.round(F.col("unit_cost") * (F.lit(1.25) + (F.col("product_id") % 7) / 20), 2))
    .withColumn("is_active", (F.col("product_id") % 17 != 0))
)

customers.write.mode("overwrite").format("delta").saveAsTable(f"{SILVER}.customers")
products.write.mode("overwrite").format("delta").saveAsTable(f"{SILVER}.products")

print("Silver customers:", customers.count())
print("Silver products :", products.count())

## 2. Orders and controlled quality issues

The dataset intentionally includes bad rows. They are visible enough for
training, but small enough not to dominate the scenario.

In [ ]:
orders = (
    spark.range(1, 120001).withColumnRenamed("id", "order_id")
    .withColumn("customer_id", ((F.col("order_id") * 17) % 10000 + 1).cast("long"))
    .withColumn("order_date", F.date_add(F.to_date(F.lit("2024-01-01")), (F.col("order_id") % 730).cast("int")))
    .withColumn("channel", F.expr("element_at(array('Online','Retail','Partner'), cast(order_id % 3 as int) + 1)"))
    .withColumn(
        "status",
        F.when(F.col("order_id") % 101 == 0, "Unknown")
         .when(F.col("order_id") % 13 == 0, "Returned")
         .when(F.col("order_id") % 11 == 0, "Cancelled")
         .otherwise("Completed")
    )
    .withColumn("region", F.expr("element_at(array('North','South','East','West'), cast(order_id % 4 as int) + 1)"))
)

# Controlled future-date issue.
orders = orders.withColumn(
    "order_date",
    F.when(F.col("order_id") % 997 == 0, F.date_add(F.current_date(), 30)).otherwise(F.col("order_date"))
)

orders.write.mode("overwrite").format("delta").saveAsTable(f"{SILVER}.sales_orders")

line_base = spark.range(1, 360001).withColumnRenamed("id", "line_id")
order_lines = (
    line_base
    .withColumn("order_id", ((F.col("line_id") - 1) % 120000 + 1).cast("long"))
    .withColumn("product_id", ((F.col("line_id") * 19) % 500 + 1).cast("long"))
    .withColumn("quantity", ((F.col("line_id") % 5) + 1).cast("int"))
    .join(products.select("product_id", "unit_price", "unit_cost", "category"), "product_id", "left")
    .join(orders.select("order_id", "customer_id", "order_date", "channel", "status", "region"), "order_id", "left")
    .withColumn("discount_pct", F.when(F.col("line_id") % 23 == 0, F.lit(0.15)).otherwise(F.lit(0.0)))
    .withColumn("unit_price", F.when(F.col("line_id") % 1543 == 0, F.lit(None).cast("double")).otherwise(F.col("unit_price")))
    .withColumn("line_revenue", F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1) - F.col("discount_pct")), 2))
    .withColumn("line_cost", F.round(F.col("quantity") * F.col("unit_cost"), 2))
    .withColumn("line_margin", F.round(F.col("line_revenue") - F.col("line_cost"), 2))
)

# Controlled duplicate issue.
duplicates = order_lines.filter("line_id % 10007 = 0").withColumn("line_id", F.col("line_id") + F.lit(10000000))
order_lines = order_lines.unionByName(duplicates)

order_lines.write.mode("overwrite").format("delta").saveAsTable(f"{SILVER}.order_lines")

print("Silver sales_orders:", orders.count())
print("Silver order_lines :", order_lines.count())

## 3. Starter Gold objects

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_date AS
SELECT
  explode(sequence(to_date('2024-01-01'), to_date('2026-12-31'), interval 1 day)) AS date_key
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_date AS
SELECT
  date_key,
  year(date_key) AS year,
  month(date_key) AS month,
  date_format(date_key, 'yyyy-MM') AS year_month,
  quarter(date_key) AS quarter
FROM {GOLD}.dim_date
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_product AS
SELECT product_id, product_name, category, subcategory, unit_cost, unit_price, is_active
FROM {SILVER}.products
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_customer AS
SELECT customer_id, customer_name, city, region, country, segment, created_date
FROM {SILVER}.customers
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_sales AS
SELECT
  l.line_id,
  l.order_id,
  l.order_date,
  l.customer_id,
  l.product_id,
  l.channel,
  l.status,
  l.region,
  l.quantity,
  l.unit_price,
  l.unit_cost,
  l.discount_pct,
  l.line_revenue,
  l.line_cost,
  l.line_margin,
  l.category
FROM {SILVER}.order_lines l
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.kpi_daily AS
SELECT
  order_date,
  SUM(CASE WHEN status = 'Completed' THEN line_revenue ELSE 0 END) AS revenue,
  SUM(CASE WHEN status = 'Completed' THEN line_margin ELSE 0 END) AS gross_margin,
  COUNT(DISTINCT CASE WHEN status = 'Completed' THEN order_id END) AS completed_orders,
  COUNT(DISTINCT CASE WHEN status = 'Returned' THEN order_id END) AS returned_orders
FROM {GOLD}.fact_sales
GROUP BY order_date
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.revenue_monthly AS
SELECT
  date_format(order_date, 'yyyy-MM') AS year_month,
  region,
  category,
  SUM(CASE WHEN status = 'Completed' THEN line_revenue ELSE 0 END) AS revenue,
  SUM(CASE WHEN status = 'Completed' THEN line_margin ELSE 0 END) AS gross_margin
FROM {GOLD}.fact_sales
GROUP BY date_format(order_date, 'yyyy-MM'), region, category
""")

print("[OK] Starter Gold objects created")

In [ ]:
spark.sql(f"SHOW TABLES IN {GOLD}").show(truncate=False)